In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os

# --- CONFIGURATION ---
radiomics_file = '../Results/final_radiomics_merged.csv' # Public vs Square (Task 36)
dosiomics_file = '../Results/final_local_dosiomics.csv'  # Ahsania vs Square (Task 33)
output_dir = '../Results/Final_Figures/'

os.makedirs(output_dir, exist_ok=True)

print("--- GENERATING FINAL THESIS FIGURES ---")

# 1. FIGURE 4.1: RADIOMICS DOMAIN SHIFT (PCA)
if os.path.exists(radiomics_file):
    print("Generating Figure 4.1 (Radiomics PCA)...")
    df_rad = pd.read_csv(radiomics_file)
    
    # We need to re-run PCA quickly to get the coordinates if they aren't saved
    # Or assuming Task 36 saved the PCA coordinates. 
    # If Task 36 saved "final_radiomics_merged.csv" with raw features, we re-calc PCA here for the plot.
    from sklearn.preprocessing import StandardScaler
    from sklearn.decomposition import PCA
    
    features = [c for c in df_rad.columns if 'original_' in c]
    X = df_rad[features].fillna(0)
    y = df_rad['Cohort']
    
    pca = PCA(n_components=2)
    X_scaled = StandardScaler().fit_transform(X)
    pcs = pca.fit_transform(X_scaled)
    
    df_pca = pd.DataFrame(data=pcs, columns=['PC1', 'PC2'])
    df_pca['Cohort'] = y.values
    
    fig_pca = px.scatter(
        df_pca, x='PC1', y='PC2', color='Cohort', 
        symbol='Cohort', opacity=0.7,
        color_discrete_map={'Public (Western)': '#3498db', 'Local (Square Hospital)': '#e74c3c'},
        title="<b>Figure 4.1: Radiomic Domain Shift</b><br><sup>Visual separation of Western vs. Bangladeshi imaging phenotypes</sup>"
    )
    fig_pca.update_layout(template='plotly_white', width=800, height=600)
    fig_pca.write_image(f"{output_dir}Figure_4.1_Radiomics_PCA.png", scale=3)
    print("  -> Saved Figure 4.1")

# 2. FIGURE 4.2: DOSIOMICS AUDIT (BOXPLOT)
if os.path.exists(dosiomics_file):
    print("Generating Figure 4.2 (Dosiomics Audit)...")
    df_dos = pd.read_csv(dosiomics_file)
    
    fig_box = px.box(
        df_dos, x='Source', y='D95_Gy', color='Source',
        points='all',
        color_discrete_map={'Ahsania': '#e74c3c', 'Square': '#3498db'},
        title="<b>Figure 4.2: Tumor Coverage Audit</b><br><sup>Comparison of Target Dose (D95) between centers</sup>"
    )
    fig_box.update_layout(
        template='plotly_white', width=800, height=600, showlegend=False,
        xaxis_title="Hospital Center", yaxis_title="Dose (Gy) covering 95% of Target"
    )
    fig_box.write_image(f"{output_dir}Figure_4.2_Dosiomics_Boxplot.png", scale=3)
    print("  -> Saved Figure 4.2")

# 3. FIGURE 4.3: FEATURE IMPORTANCE (BAR CHART)
# We will use the 'domain_shift_stats.csv' if available, or generate a simple one
stats_file = '../Results/domain_shift_stats.csv'
if os.path.exists(stats_file):
    print("Generating Figure 4.3 (Top Divergent Features)...")
    df_stats = pd.read_csv(stats_file).head(10).sort_values(by='Fold_Change', ascending=True)
    
    # Clean names
    df_stats['Feature_Clean'] = df_stats['Feature'].str.replace('original_', '').str.replace('glcm_', '').str[:25]
    
    fig_bar = px.bar(
        df_stats, x='Fold_Change', y='Feature_Clean', orientation='h',
        color='P_Value',
        title="<b>Figure 4.3: Drivers of Domain Shift</b><br><sup>Features with highest fold-change (Local / Public)</sup>"
    )
    fig_bar.update_layout(
        template='plotly_white', width=800, height=600,
        xaxis_title="Fold Change (Ratio)", yaxis_title="Radiomic Feature"
    )
    fig_bar.write_image(f"{output_dir}Figure_4.3_Feature_Importance.png", scale=3)
    print("  -> Saved Figure 4.3")

print("\nAll figures generated successfully. You are ready to write Chapter 4.")

--- GENERATING FINAL THESIS FIGURES ---
Generating Figure 4.1 (Radiomics PCA)...
  -> Saved Figure 4.1
Generating Figure 4.2 (Dosiomics Audit)...
  -> Saved Figure 4.2
Generating Figure 4.3 (Top Divergent Features)...
  -> Saved Figure 4.3

All figures generated successfully. You are ready to write Chapter 4.
